In [1]:
import pandas as pd
import requests
from tqdm import tqdm

from itertools import product
import time

from typing import Union


pd.set_option('display.max_columns', None)

In [5]:
df = pd.read_parquet('data/data_2018.parquet')

In [6]:
df

,id,date,time,lat,lng,dtp_type,road_title,weather,road_cond,car_mark,car_model,color,car_year,driver_gender,driver_exp,driver_safety_belt,driver_alco,driver_trauma,driver_violations
0,,31.01.2018,10:05,53.377249,83.938186,Столкновение,,[Ясно],Сухое,ВАЗ,Жигули ВАЗ-2107 модификации,Синий,2008,Мужской,5,Нет,,"Раненый, находящийся (находившийся) на амбула...",[Выезд на полосу встречного движения]
1,,31.01.2018,10:05,53.377249,83.938186,Столкновение,,[Ясно],Сухое,TOYOTA,Vista,Серый,1997,Мужской,7,Нет,,Не пострадал,[Нет нарушений]
2,,31.01.2018,14:55,52.8761,80.7975,Столкновение,Ребриха - Шарчино - Корчино - Завьялово - Лень...,[Ясно],Обработанное противогололедными материалами,TOYOTA,Corolla,Серый,2006,Мужской,12,Нет,,"Раненый, находящийся (находившийся) на стацион...",[Выезд на полосу встречного движения]
3,,31.01.2018,14:55,52.8761,80.7975,Столкновение,Ребриха - Шарчино - Корчино - Завьялово - Лень...,[Ясно],Обработанное противогололедными материалами,TOYOTA,Premio,Серый,2003,Мужской,16,Нет,,"Раненый, находящийся (находившийся) на амбула...",[Нет нарушений]
4,,31.01.2018,16:50,52.540534,85.199994,Наезд на пешехода,,[Ясно],Со снежным накатом,LEXUS,RX,Черный,2003,Мужской,22,Нет,,Не пострадал,"[Несоблюдение условий, разрешающих движение тр..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248342,,01.12.2018,10:50,57.680454,39.803424,Столкновение,,[Ясно],Сухое,ВАЗ,ВАЗ 2112 и модификации,Серый,2007,Мужской,95,Нет,,Не пострадал,"[Выезд на полосу встречного движения в местах,..."
248343,,01.12.2018,20:40,57.72287,39.750777,Столкновение,ЯРОСЛАВЛЬ-РЫБИНСК,[Ясно],Сухое,VOLKSWAGEN,Passat,Иные цвета,1994,Мужской,95,Нет,24,"Раненый, находящийся (находившийся) на амбула...",[Неправильный выбор дистанции]
248344,,01.12.2018,20:40,57.72287,39.750777,Столкновение,ЯРОСЛАВЛЬ-РЫБИНСК,[Ясно],Сухое,ВАЗ,Жигули ВАЗ-2107 модификации,Синий,2004,Мужской,26,Нет,,Не пострадал,[Нет нарушений]
248345,,01.12.2018,18:00,57.693432,39.772385,Наезд на пешехода,,[Ясно],Сухое,,,,,Не определен,,Нет,,Не пострадал,[Нет нарушений]


In [2]:
# получение справочника регионов
regions_url = 'http://стат.гибдд.рф/opendataapi/v1/dictionary/rows?code=1'

responce = requests.get(regions_url)
regions = pd.DataFrame()

if responce.status_code == 200:
    responce_json = responce.json()
    regions = pd.DataFrame(responce_json['results'][0]['dict_rows'])[['rows_code', 'rows_name']]
    regions.rename(columns={'rows_code': 'code', 'rows_name': 'region_name'}, inplace=True)

regions.head()


,code,region_name
0,1101,Алтайский край
1,1110,Амурская область
2,1111,Архангельская область
3,1112,Астраханская область
4,1114,Белгородская область


In [4]:
regions.sort_values(by='code')

,code,region_name
65,1100,Российская Федерация
0,1101,Алтайский край
84,1102,Херсонская область
27,1103,Краснодарский край
28,1104,Красноярский край
...,...,...
86,1196,Чеченская Республика
87,1197,Чувашская Республика - Чувашия
60,1198,Республика Саха (Якутия)
14,1199,Еврейская автономная область


In [23]:
test_url = 'http://стат.гибдд.рф/opendataapi/v1/kartdtp/rows?dat=1.2026&reg=1114&pok=1'

responce = requests.get(test_url)
test_json = responce.json()

In [ ]:
def parse_data(data) -> Union[pd.DataFrame, None]:
    try:
        dtp_cards = data['results']['region_list'][0]['pok_list'][0]['result'][0]['dtpcardlist']['info_dtp']
    except:
        return None

    parsed_info = []
    error_parse = 0
    
    for dtp_card in dtp_cards:
        try:
            base_info = {
                'id': dtp_card.get('empt_number'),
                'date': dtp_card.get('date_dtp'),
                'time': dtp_card.get('time'),
                'lat': dtp_card.get('coord_w'),
                'lng': dtp_card.get('coord_l'),
                'dtp_type': dtp_card.get('dtpv'),
                'road_title': dtp_card.get('dor')
            }

            # информация о погоде
            weather_dict = dtp_card.get('dor_usl', {})

            base_info['weather'] = weather_dict.get('spog')
            base_info['road_cond'] = weather_dict.get('s_pch')
            
            # информация об участниках
            for participant in dtp_card.get('ts_info', []):
                participant_info = base_info.copy()
                participant_info['car_mark'] = participant['marka_ts']
                participant_info['car_model'] = participant['m_ts']
                participant_info['color'] = participant['color']
                participant_info['car_year'] = participant['g_v']

                # информация о водителе
                for passenger in participant.get('ts_uch', []):
                    if passenger['kt_uch'] == 'Водитель':
                        participant_info['driver_gender'] = passenger['pol']
                        participant_info['driver_exp'] = passenger['v_st']
                        participant_info['driver_safety_belt'] = passenger['safety_belt']
                        participant_info['driver_alco'] = passenger['alco']
                        participant_info['driver_trauma'] = passenger['s_t']
                        participant_info['driver_violations'] = passenger['npdd']
                    
                        parsed_info.append(participant_info)

        except:
            error_parse += 1

    print(f"Ошибок при обработке: {error_parse}")

    return pd.DataFrame(parsed_info)



df = parse_data(test_json)
df

100%|██████████| 78/78 [00:00<00:00, 16332.47it/s]

Ошибок при обработке: 0


,id,date,time,lat,lng,dtp_type,road_title,weather,road_cond,car_mark,car_model,color,car_year,driver_gender,driver_exp,driver_safety_belt,driver_alco,driver_trauma,driver_violations
0,140001017,31.01.2026,21:30,50.533611,36.574167,Наезд на стоящее ТС,,[Ясно],Мокрое,KIA,K5,Белый,2021,Мужской,14,Да,00,Не пострадал,[Нарушение правил расположения ТС на проезжей ...
1,140001017,31.01.2026,21:30,50.533611,36.574167,Наезд на стоящее ТС,,[Ясно],Мокрое,RENAULT,Logan,Белый,2011,Мужской,4,Да,00,"Раненый, находящийся (находившийся) на амбулат...",[Нет нарушений]
2,140001017,31.01.2026,21:30,50.533611,36.574167,Наезд на стоящее ТС,,[Ясно],Мокрое,RENAULT,Logan,Белый,2014,Мужской,23,Да,00,Не пострадал,[Нет нарушений]
3,140001013,31.01.2026,16:40,51.082343,36.298259,Съезд с дороги,М-2 Крым Москва - Тула - Орел - Курск - Белгор...,[Ясно],Свежеуложенная поверхностная обработка,ВАЗ,Largus (Ларгус),Серый,2019,Мужской,10,Нет,00,Скончался на месте ДТП по прибытию скорой меди...,[Несоответствие скорости конкретным условиям д...
4,140000945,30.01.2026,10:15,50.595151,36.605613,Наезд на пешехода,,[Снегопад],Заснеженное,MAZDA,CX-5,Синий,2021,Женский,26,Да,00,Не пострадал,[Непредоставление преимущества в движении пеше...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,140000085,03.01.2026,21:50,50.601749,36.594837,Наезд на пешехода,,[Пасмурно],Мокрое,TOYOTA,Avensis,Серый,2005,Мужской,15,Да,00,Не пострадал,[Нет нарушений]
117,140000056,02.01.2026,17:45,50.572063,36.569946,Наезд на пешехода,,[Пасмурно],Мокрое,ВАЗ,Vesta (Веста),Коричневый,2021,Мужской,37,Да,00,Не пострадал,[Нарушение правил проезда пешеходного перехода]
118,140000061,02.01.2026,21:50,50.736150,37.938420,Съезд с дороги,"М-2 Крым, соединительная дорога Белгород - М-4...",[Пасмурно],Обработанное противогололедными материалами,ВАЗ,Granta (Гранта),Черный,2013,Мужской,1,Да,00,"Раненый, находящийся (находившийся) на стацион...",[Несоответствие скорости конкретным условиям д...
119,140000532,02.01.2026,18:00,51.320213,37.261666,Наезд на пешехода,,[Ясно],Сухое,,,,,Не определен,,Нет,00,Не пострадал,[Нарушение правил расположения ТС на проезжей ...


In [ ]:
years = range(2016, 2026)
months = range(1, 13)
regions_code = regions['code'].tolist()
BASE_URL = 'http://стат.гибдд.рф/opendataapi/v1/kartdtp/rows?'

for year in years:
    year_df = []
    for month in months:

        for code in tqdm(regions_code, desc=f'Обработка за {month}.{year}'):
            try:
                params = {
                    'dat': f'{month}.{year}',
                    'reg': code,
                    'pok': '1'
                }

                response = requests.get(BASE_URL, params, timeout=30)
                response.raise_for_status()        
                response_json = response.json()

                month_region_df = parse_data(responce_json)
                if month_region_df is not None:
                    year_df.append(month_region_df)
                else:
                    print(f'❌ Пропуск региона {code}')
            except:
                print(f'❌ Пропуск региона {code}')
            time.sleep(1)

    combined_df = pd.concat(year_df, ignore_index=True)
    combined_df.to_parquet(f'data/data_{year}.parquet')

100%|██████████| 173/173 [00:00<?, ?it/s]1 [00:00<?, ?it/s]


Ошибок при обработке: 0


100%|██████████| 173/173 [00:00<?, ?it/s]1 [00:01<02:31,  1.69s/it]


Ошибок при обработке: 0


100%|██████████| 173/173 [00:00<00:00, 11032.27it/s]:09,  1.45s/it]


...

Ошибок при обработке: 0


100%|██████████| 173/173 [00:00<00:00, 86764.87it/s]:03,  1.51s/it]


Ошибок при обработке: 0


Обработка за 1.2016:  11%|█         | 10/91 [00:17<02:22,  1.76s/it]


HTTPError: 500 Server Error:  for url: http://xn--80a7adb.xn--90adear.xn--p1ai/opendataapi/v1/kartdtp/rows?dat=1.2016&reg=1145&pok=1